# 7. A Distribution Problem

In [2]:
import pulp as pl

def solve(model):
    model.solve(pl.PULP_CBC_CMD(msg=False))
    if pl.LpStatus[model.status] != "Optimal":
        raise RuntimeError(pl.LpStatus[model.status])

production_cost = [2, 1, 1.8]
capacity = [4500, 9000, 11250]
demand = [2700, 2700, 9000, 4500, 3600]
transport = [[.05,.07,.11,.15,.16], [.08,.06,.10,.12,.15], [.10,.09,.09,.10,.16]]
model = pl.LpProblem("Distribution", pl.LpMaximize)
x = {(i,j): pl.LpVariable(f"ship_{i+1}_{j+1}", lowBound=0) for i in range(3) for j in range(5)}
model += pl.lpSum((2.5-production_cost[i]-transport[i][j])*x[i,j] for i,j in x)
for i in range(3):
    model += pl.lpSum(x[i,j] for j in range(5)) <= capacity[i]
for j in range(5):
    model += pl.lpSum(x[i,j] for i in range(3)) == demand[j]
solve(model)
print("Plant      W1      W2      W3      W4      W5")
for i in range(3):
    print(f"{i+1:5}" + "".join(f"{x[i,j].value():8.0f}" for j in range(5)))
print(f"Maximum monthly profit: USD {pl.value(model.objective):,.2f}")

Plant      W1      W2      W3      W4      W5
    1    2250       0       0       0       0
    2     450    2700    2250       0    3600
    3       0       0    6750    4500       0
Maximum monthly profit: USD 20,367.00


Optimal shipments to wholesalers 1–5:

- Plant 1: **2,250; 0; 0; 0; 0**
- Plant 2: **450; 2,700; 2,250; 0; 3,600**
- Plant 3: **0; 0; 6,750; 4,500; 0**

Maximum monthly profit: **$20,367**.